# 01 — Data Exploration

Load and explore Bybit L2 order book data. Visualize raw book snapshots,
check data quality, and understand the structure of the dataset.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as sp_stats

from dashboard._mock_data import generate_snapshots

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.6f}".format)

## 1.1 Load Data

We generate realistic synthetic L2 snapshots matching the schema produced by
`book_reconstructor.reconstruct()`. To use real data, replace with:
```python
from src.data_loader import load_directory
from src.book_reconstructor import reconstruct
events = load_directory("data/raw", symbol="BTCUSDT")
snapshots = reconstruct(events)
```

In [ ]:
df = generate_snapshots(n_timestamps=7200, start="2025-01-15 09:00:00", freq="1s")
print(f"Shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
print(f"Time range: {df['timestamp'].iloc[0]} to {df['timestamp'].iloc[-1]}")
df.head()

## 1.2 Summary Statistics

In [ ]:
summary_cols = ["mid_price", "spread", "bid_price_1", "ask_price_1",
                "bid_qty_1", "ask_qty_1", "last_trade_price", "last_trade_qty"]
df[summary_cols].describe()

In [ ]:
print("=== Data Quality Report ===")
print(f"Total rows: {len(df):,}")
nan_counts = df.isna().sum()
print(f"NaN count: {nan_counts.sum()} total across all columns")
print(f"\nMid-price range: ${df['mid_price'].min():,.2f} — ${df['mid_price'].max():,.2f}")
print(f"Mid-price mean:  ${df['mid_price'].mean():,.2f}")
print(f"Spread mean:     {df['spread'].mean():.4f} USD")
print(f"Spread median:   {df['spread'].median():.4f} USD")
print(f"\nTrade side distribution:")
print(df["last_trade_side"].value_counts())

## 1.3 Mid-Price and Spread Time Series

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Mid-Price (BTCUSDT)", "Spread (USD)"],
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=df["timestamp"], y=df["mid_price"],
                         mode="lines", name="Mid Price",
                         line=dict(color="#3498db", width=1)), row=1, col=1)

fig.add_trace(go.Scatter(x=df["timestamp"], y=df["spread"],
                         mode="lines", name="Spread",
                         line=dict(color="#e74c3c", width=1)), row=2, col=1)

fig.update_layout(height=500, title_text="Price and Spread Overview",
                  showlegend=False, template="plotly_dark")
fig.update_yaxes(title_text="USD", row=1, col=1)
fig.update_yaxes(title_text="USD", row=2, col=1)
fig.show()

## 1.4 Order Book Shape Analysis

Volume distribution across the top 10 price levels.

In [ ]:
bid_means = [df[f"bid_qty_{i}"].mean() for i in range(1, 11)]
ask_means = [df[f"ask_qty_{i}"].mean() for i in range(1, 11)]
levels = list(range(1, 11))

fig = go.Figure()
fig.add_trace(go.Bar(x=levels, y=bid_means, name="Bid (avg)",
                     marker_color="#2ecc71", opacity=0.8))
fig.add_trace(go.Bar(x=levels, y=ask_means, name="Ask (avg)",
                     marker_color="#e74c3c", opacity=0.8))

fig.update_layout(title="Average Volume by Book Level",
                  xaxis_title="Level (1 = best)", yaxis_title="Avg Quantity (BTC)",
                  barmode="group", template="plotly_dark", height=400)
fig.show()

In [ ]:
print("=== Volume Decay Profile ===")
print(f"{'Level':>6} {'Bid Vol':>10} {'Ask Vol':>10} {'Bid/L1':>10} {'Ask/L1':>10}")
print("-" * 50)
for i in range(1, 11):
    bv = df[f"bid_qty_{i}"].mean()
    av = df[f"ask_qty_{i}"].mean()
    print(f"{i:>6} {bv:>10.3f} {av:>10.3f} {bv/bid_means[0]:>10.2%} {av/ask_means[0]:>10.2%}")

## 1.5 Spread Distribution

In [ ]:
spread_bps = (df["ask_price_1"] - df["bid_price_1"]) / df["mid_price"] * 10000

fig = go.Figure()
fig.add_trace(go.Histogram(x=spread_bps, nbinsx=80, name="Spread",
                           marker_color="#9b59b6", opacity=0.8))
fig.add_vline(x=spread_bps.median(), line_dash="dash", line_color="white",
              annotation_text=f"Median: {spread_bps.median():.2f} bps")

fig.update_layout(title="Spread Distribution (basis points)",
                  xaxis_title="Spread (bps)", yaxis_title="Count",
                  template="plotly_dark", height=400)
fig.show()

print(f"Spread statistics (bps):")
print(f"  Mean:   {spread_bps.mean():.2f}")
print(f"  Median: {spread_bps.median():.2f}")
print(f"  Std:    {spread_bps.std():.2f}")
print(f"  Skew:   {spread_bps.skew():.2f}")

## 1.6 Order Book Snapshot Visualization

In [ ]:
snapshot_idx = len(df) // 2
row = df.iloc[snapshot_idx]

bid_prices = [row[f"bid_price_{i}"] for i in range(1, 11)]
bid_qtys = [row[f"bid_qty_{i}"] for i in range(1, 11)]
ask_prices = [row[f"ask_price_{i}"] for i in range(1, 11)]
ask_qtys = [row[f"ask_qty_{i}"] for i in range(1, 11)]

fig = go.Figure()
fig.add_trace(go.Bar(x=bid_prices, y=bid_qtys, name="Bids",
                     marker_color="#2ecc71", width=0.3, opacity=0.8))
fig.add_trace(go.Bar(x=ask_prices, y=ask_qtys, name="Asks",
                     marker_color="#e74c3c", width=0.3, opacity=0.8))
fig.add_vline(x=row["mid_price"], line_dash="dash", line_color="gold",
              annotation_text=f"Mid: ${row['mid_price']:,.2f}")

fig.update_layout(title=f"Order Book Snapshot at {row['timestamp']}",
                  xaxis_title="Price (USD)", yaxis_title="Quantity (BTC)",
                  template="plotly_dark", height=400)
fig.show()

## 1.7 Return Statistics

In [ ]:
log_ret = np.log(df["mid_price"] / df["mid_price"].shift(1)).dropna()

print("=== 1-Second Log Return Statistics ===")
print(f"  Mean:     {log_ret.mean():.8f}")
print(f"  Std:      {log_ret.std():.8f}")
print(f"  Skewness: {sp_stats.skew(log_ret):.4f}")
print(f"  Kurtosis: {sp_stats.kurtosis(log_ret):.4f} (excess)")
print(f"  Min:      {log_ret.min():.8f}")
print(f"  Max:      {log_ret.max():.8f}")
print(f"  Jarque-Bera p-value: {sp_stats.jarque_bera(log_ret).pvalue:.6f}")

fig = go.Figure()
fig.add_trace(go.Histogram(x=log_ret, nbinsx=100, name="Returns",
                           marker_color="#3498db", opacity=0.7))
fig.update_layout(title="1-Second Log Return Distribution",
                  xaxis_title="Log Return", yaxis_title="Count",
                  template="plotly_dark", height=400)
fig.show()

## 1.8 Top-of-Book Dynamics

In [ ]:
imb = (df["bid_qty_1"] - df["ask_qty_1"]) / (df["bid_qty_1"] + df["ask_qty_1"])

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Top-of-Book Imbalance", "Top-Level Volume"],
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=df["timestamp"], y=imb, mode="lines",
                         name="Imbalance", line=dict(color="#f39c12", width=0.5)),
              row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)

fig.add_trace(go.Scatter(x=df["timestamp"], y=df["bid_qty_1"], mode="lines",
                         name="Bid L1", line=dict(color="#2ecc71", width=0.5)),
              row=2, col=1)
fig.add_trace(go.Scatter(x=df["timestamp"], y=df["ask_qty_1"], mode="lines",
                         name="Ask L1", line=dict(color="#e74c3c", width=0.5)),
              row=2, col=1)

fig.update_layout(height=500, template="plotly_dark",
                  title_text="Top-of-Book Dynamics")
fig.show()

## Summary

Key observations from data exploration:

1. **Data shape:** Each snapshot contains 10 levels of bid/ask prices and quantities, plus mid-price, spread, and last trade info (43 columns) at 1-second resolution.
2. **Volume decay:** Book volume decays roughly as 1/level with distance from the best price, consistent with typical LOB structure in liquid futures markets.
3. **Spread distribution:** Right-skewed with a concentrated mode near the median, and a tail of wider spreads during volatile periods.
4. **Returns:** Near-zero mean with excess kurtosis (fat tails), typical of high-frequency financial returns.
5. **Book imbalance:** Fluctuates rapidly around zero, providing a noisy but informative directional pressure signal.